[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/AcademiXBase/deep-learning-from-scratch/blob/dev/my_notebooks/ch03_02.ipynb)

In [ ]:
# Colab で実行している場合、リポジトリをクローンする
#!git clone -b dev https://github.com/AcademiXBase/deep-learning-from-scratch.git
#%cd deep-learning-from-scratch/my_notebooks

In [ ]:
from __future__ import annotations

import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
import sympy as sp
from sympy import Matrix, latex
from IPython.display import Math, Markdown, display
from PIL import Image

from dataset.mnist import load_mnist
from common.functions import sigmoid, softmax
import pickle

## 多次元配列の計算

ニューラルネットワークの計算を効率的に行うためには、NumPy の多次元配列 (テンソル) の取り扱いを理解することが重要です。

### 多次元配列

**多次元配列の定義**  

多次元配列とは、数字が 1 列に並んだもの (ベクトル)、長方形状に並んだもの (行列)、さらには N 次元状に並べたもの (テンソル) など、数字の集まりのことを指します。

In [ ]:
A = np.array([1, 2, 3, 4])

**次元数と形状の確認**  

* 配列の次元数は `np.ndim()` 関数で取得できます。  

* 配列の形状は、インスタンス変数である `shape` からタプルとして取得できます。  
  例えば、2 次元配列なら (行数、列数) となります。  

In [ ]:
print(f"A = {A}")  # 配列の表示
print(f"A.ndim = {A.ndim}")  # 次元数
print(f"A.shape = {A.shape}")  # 形状
print(f"A.shape[0] = {A.shape[0]}")  # 1次元目の要素数
print(f"A.size = {A.size}")  # 要素数
print(f"A.dtype = {A.dtype}")  # データ型

**次元の呼び方**  

2 次元配列は「行列 (matrix)」と呼ばれ、その横方向の並びを行 (row)、縦方向の並びを列（column）と呼びます。

**テンソル**

1 次元配列 (ベクトル) や 2 次元配列 (行列) をまとめて「テンソル」と呼びます。

In [ ]:
B = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
])

In [ ]:
print(f"B = \n{B}")  # 配列の表示
print(f"B.ndim = {B.ndim}")  # 次元数
print(f"B.shape = {B.shape}")  # 形状
print(f"B.shape[0] = {B.shape[0]}")  # 1次元目の要素数
print(f"B.size = {B.size}")  # 要素数
print(f"B.dtype = {B.dtype}")  # データ型

### 行列の積

ニューラルネットワークの層間の信号伝達は、行列を用いることでまとめて効率的に計算できます。

Sympy での書き方

In [ ]:
def sym_matrix(
    base: str,
    n_rows: int,
    n_cols: int,
    layer: Optional[int] = None,
    bold: bool = True,
) -> Matrix:
    """
    a_{ij}^{(layer)} のような記号行列を作る (1-indexed)

    - base: "x", "w", "b", ...
    - layer: None なら上付き無し、整数なら ^{(layer)} を付ける
    """
    sup = "" if layer is None else rf"^{{({layer})}}"
    return Matrix(
        n_rows,
        n_cols,
        lambda i, j: sp.Symbol(rf"{base}_{{{i+1}{j+1}}}{sup}"),
    )

def sym_vector(base: str, length: int, *, layer: Optional[int] = None) -> Matrix:
    """x_i^{(layer)} のような 1 x L 行ベクトルを作る (1-indexed)"""
    sup = "" if layer is None else rf"^{{({layer})}}"
    elems = [sp.Symbol(rf"{base}_{{{k}}}{sup}") for k in range(1, length + 1)]
    return Matrix([elems])

In [ ]:
def latex_matrix(M: Matrix, style: str = "none") -> str:
    """
    SymPy 行列を LaTeX 文字列に変換

    style:
      - "paren": ( )
      - "bracket": [ ]
      - "none": delimiter無し
    """
    from sympy import latex as _latex

    if style == "paren":
        return _latex(M, mat_str="pmatrix")
    if style == "bracket":
        return _latex(M, mat_str="bmatrix")
    if style == "none":
        return _latex(M, mat_str="pmatrix", mat_delim="")
    raise ValueError("style must be 'paren'/'bracket'/'none'")

In [ ]:
def sigmoid_sym(M: Matrix) -> Matrix:
    """SymPy 行列に対する sigmoid"""
    return M.applyfunc(lambda t: 1 / (1 + sp.exp(-t)))


def identity_sym(M: Matrix) -> Matrix:
    """SymPy 行列に対する恒等写像"""
    return M

In [ ]:
def make_subs_disp(subs_num: Dict[sp.Expr, float], fmt: str = "{:g}") -> Dict[sp.Expr, sp.Symbol]:
    """
    「数値代入をするが、右辺の式は評価させない」ための置換辞書を作る

    例) x1 -> Symbol("1.0") のように置くことで、式は 1.0*w + ... の形で残る
    """
    subs_disp: Dict[sp.Expr, sp.Symbol] = {}
    for k, v in subs_num.items():
        if isinstance(v, (float, int, np.floating, np.integer)):
            subs_disp[k] = sp.Symbol(fmt.format(float(v)))
        else:
            # 念のため: 文字列化して Symbol にする
            subs_disp[k] = sp.Symbol(str(v))
    return subs_disp


In [ ]:
A = sym_matrix("a", 2, 3)  # a_{11}, a_{12}, ...
B = sym_matrix("b", 3, 2)

C = A*B

In [ ]:
Math(rf"{latex_matrix(A)}\,{latex_matrix(B)} = {latex_matrix(C)}")

In [ ]:
A_num = sp.Matrix([
    [1, 2, 3],
    [4, 5, 6]
    ])

B_num = sp.Matrix([
    [1, 2],
    [3, 4],
    [5, 6]
    ])

C_num = A_num * B_num

In [ ]:
Math(rf"$${latex_matrix(A_num)}\,{latex_matrix(B_num)} = {latex_matrix(C_num)}$$")

In [ ]:
display(Markdown(
    rf"""
$$
\mathrm{{shape}}(A)=({A_num.rows},{A_num.cols}),\quad
\mathrm{{shape}}(B)=({B_num.rows},{B_num.cols}),\quad
\mathrm{{shape}}(C)=({C_num.rows},{C_num.cols})
$$
"""
))

Numpy での書き方  

行列の積は、`np.dot()` 関数を用いて計算できます。  

行列の積を計算するには、2 つの行列で「対応する次元の要素数」を一致させる必要があります。  

例えば、行列 A (M×K) と行列 B (K×N) の積を取る場合、A の第 1 次元の要素数 (K) と B の第 0 次元の要素数 (K) が一致している必要があります。計算結果は、行列 A の行数 (M) と行列 B の列数 (N) から構成されます (M×N)。

In [ ]:
A_num = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

B_num = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
])

C_num = A_num.dot(B_num)

In [ ]:
# 配列の表示
print(f"A = \n{A_num}\n")
print(f"B = \n{B_num}\n")
print(f"C = \n{C_num}")

`np.dot()` を使うことで、たとえ出力の要素数が 100 や 1000 であっても、一度の演算でまとめて結果を計算できます。

## 3 層ニューラルネットワークの実装

この節では、多次元配列による行列演算を活用し、3 層からなるニューラルネットワークの推論処理 (入力から出力への処理) である順伝播 (Forward Propagation) を実装します。

### 各層における信号伝達の実装

各層の信号伝達は重み付き信号とバイアスの総和を計算し、それを活性化関数で変換するという2段階の処理で構成されます。  

重み付き和の計算は、入力 ($\mathrm{x}$) と重み ($\mathrm{W}$) の行列の積にバイアス ($\mathrm{b}$) を加える形で書けます。具体的には、$\mathrm{a^{(1)} = x \, W^{(1)} + b^{(1)}}$ のように表せます。  

ここでは、入力層から出力層まで、信号がどのように伝達されるかを実装します。実装例は、入力層 (第 0 層)、第 1 層 (隠れ層)、第 2 層 (隠れ層)、出力層 (第 3 層) からなるネットワークを想定しています。  

**(1) 第 0 層 (入力層) から第 1 層へ：**  

入力信号 $\mathrm{x}$ と第 1 層の重み $\mathrm{W1}$、バイアス $\mathrm{b1}$ を用いて、重み付き和 $\mathrm{a1}$ を計算し、シグモイド関数で変換します。  

* 重み付き和の計算：`a1 = np.dot(x, W1) + b1`  
* 活性化：`z1 = sigmoid(a1)`  

第1層の出力 $\mathrm{z1}$ は、次の第 2 層への入力となります。

In [ ]:
layer = 1
x = sym_vector("x", 2, layer=None)

W1 = sym_matrix("W", 2, 3, layer=layer)
b1 = sym_vector("b", 3, layer=layer)
a1 = sym_vector("a", 3, layer=layer)
z1 = sym_vector("z", 3, layer=layer)

a1_eq = x*W1 + b1

In [ ]:
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = \mathbf{{x}}\,\mathbf{{W}}^{{({layer})}} + \mathbf{{b}}^{{({layer})}}$$"))

In [ ]:
display(Markdown(rf"$$\mathbf{{x}} = {latex_matrix(x)}$$"))
display(Markdown(rf"$$\mathbf{{W}}^{{({layer})}} = {latex_matrix(W1)}$$"))
display(Markdown(rf"$$\mathbf{{b}}^{{({layer})}} = {latex_matrix(b1)}$$"))
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = {latex_matrix(a1)}$$"))

In [ ]:
subs = {
    x[0]: 1.0,  x[1]: 0.5,
    W1[0,0]: 0.1,  W1[0,1]: 0.3,  W1[0,2]: 0.5,
    W1[1,0]: 0.2,  W1[1,1]: 0.4,  W1[1,2]: 0.6,
    b1[0]: 0.1,  b1[1]: 0.2,  b1[2]: 0.3,
}

x_num = x.subs(subs)
W1_num = W1.subs(subs)
b1_num = b1.subs(subs)
a1_num = (x*W1 + b1).subs(subs)
z1_num = sigmoid_sym(a1_num)

In [ ]:
display(Markdown(rf"$$\mathbf{{x}} = {latex_matrix(x_num)}$$"))
display(Markdown(rf"$$\mathbf{{W}}^{{({layer})}} = {latex_matrix(W1_num)}$$"))
display(Markdown(rf"$$\mathbf{{b}}^{{({layer})}} = {latex_matrix(b1_num)}$$"))
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = {latex_matrix(a1_num)}$$"))
display(Markdown(rf"$$\mathbf{{z}}^{{({layer})}} = {latex_matrix(z1_num.evalf(3))}$$"))

In [ ]:
display(Markdown(rf"$${latex_matrix(a1)} = {latex_matrix(x)}\,{latex_matrix(W1)} + {latex_matrix(b1)}$$"))
display(Markdown(rf"$${latex_matrix(a1_num)} = {latex_matrix(x_num)}\,{latex_matrix(W1_num)} + {latex_matrix(b1_num)}$$"))
display(Markdown(rf"$${latex_matrix(a1_num)} = {latex_matrix(x_num * W1_num)} + {latex_matrix(b1_num)}$$"))

In [ ]:
eqs = [sp.Eq(a1[0,j], sp.simplify(a1_eq[0,j])) for j in range(a1.cols)]
display(Markdown("$$" + r"\begin{aligned}" + r"\\ ".join(latex(e) for e in eqs) + r"\end{aligned}" + "$$"))

In [ ]:
subs_disp = {k: sp.Symbol(f"{v}") for k, v in subs.items()}

A1_num = (a1_eq.subs(subs))

eqs_show = []
for j in range(A1_num.cols):
    lhs = sp.N(A1_num[0, j], 6)  # 左辺は数値
    rhs = sp.expand(a1_eq[0, j]).subs(subs_disp)  # 右辺は未評価の数値式として残す
    eqs_show.append(sp.Eq(lhs, rhs, evaluate=False))

display(Markdown(
    "$$" + r"\begin{aligned}"
    + r"\\ ".join(sp.latex(e, mul_symbol="dot") for e in eqs_show)
    + r"\end{aligned}" + "$$"
))

**(2) 第 1 層から第 2 層へ：**  

第 2 層の処理も同様に、第 1 層の出力 $\mathrm{z1}$ を入力として、重み $\mathrm{W2}$、バイアス $\mathrm{b2}$ を使って計算し、シグモイド関数で変換します。  

* 重み付き和の計算：`a2 = np.dot(z1, W2) + b2`  

* 活性化:`z2 = sigmoid(a2)`  

In [ ]:
layer = 2

W2 = sym_matrix("W", 3, 2, layer=layer)
b2 = sym_vector("b", 2, layer=layer)
a2 = sym_vector("a", 2, layer=layer)
z2 = sym_vector("z", 2, layer=layer)

a2_eq = z1*W2 + b2

In [ ]:
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = \mathbf{{z}}^{{({layer-1})}}\,\mathbf{{W}}^{{({layer})}} + \mathbf{{b}}^{{({layer})}}$$"))

In [ ]:
display(Markdown(rf"$$\mathbf{{z}}^{{({layer-1})}} = {latex_matrix(z1)}$$"))
display(Markdown(rf"$$\mathbf{{W}}^{{({layer})}} = {latex_matrix(W2)}$$"))
display(Markdown(rf"$$\mathbf{{b}}^{{({layer})}} = {latex_matrix(b2)}$$"))
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = {latex_matrix(a2_eq)}$$"))

In [ ]:
subs2 = {
    z1[0]: float(z1_num[0]), z1[1]: float(z1_num[1]), z1[2]: float(z1_num[2]),
    W2[0,0]: 0.1, W2[0,1]: 0.4,
    W2[1,0]: 0.2, W2[1,1]: 0.5,
    W2[2,0]: 0.3, W2[2,1]: 0.6,
    b2[0]: 0.1, b2[1]: 0.2,
}

W2_num = W2.subs(subs2)
b2_num = b2.subs(subs2)
a2_num = (z1*W2 + b2).subs(subs2)
z2_num = sigmoid_sym(a2_num)

In [ ]:
display(Markdown(rf"$${latex_matrix(a2)} = {latex_matrix(z1)}\,{latex_matrix(W2)} + {latex_matrix(b2)}$$"))
display(Markdown(rf"$${latex_matrix(a2_num.evalf(3))} = {latex_matrix(z1_num.evalf(3))}\,{latex_matrix(W2_num)} + {latex_matrix(b2_num)}$$"))
display(Markdown(rf"$${latex_matrix(a2_num.evalf(3))} = {latex_matrix((z1_num * W2_num).evalf(3))} + {latex_matrix(b2_num)}$$"))

In [ ]:
eqs = [sp.Eq(a2[0,j], sp.simplify(a2_eq[0,j])) for j in range(a2.cols)]
display(Markdown("$$" + r"\begin{aligned}" + r"\\ ".join(latex(e) for e in eqs) + r"\end{aligned}" + "$$"))

In [ ]:
subs_disp = {k: sp.Symbol(f"{v:.2f}") for k, v in subs2.items()}

a2_num = (a2_eq.subs(subs))

eqs_show = []
for j in range(a2_num.cols):
    lhs = sp.N(a2_num[0, j], 6)  # 左辺は数値
    rhs = sp.expand(a2_eq[0, j]).subs(subs_disp)  # 右辺は未評価の数値式として残す
    eqs_show.append(sp.Eq(lhs, rhs, evaluate=False))

display(Markdown(
    "$$" + r"\begin{aligned}"
    + r"\\ ".join(sp.latex(e, mul_symbol="dot") for e in eqs_show)
    + r"\end{aligned}" + "$$"
))

**(3) 第2層から出力層 (第3層) へ：**  

出力層の処理は、解く問題の性質に応じて活性化関数が選ばれます。ここでは、最終的な出力 $\mathrm{Y}$ を得るために、重み付き和 $\mathrm{A3}$ の計算後に恒等関数を使用しています。

* 重み付き和の計算：`A3 = np.dot(Z2, W3) + B3`  
* 活性化 (恒等関数)：`Y = identity_function(A3)`  

恒等関数は、入力をそのまま出力する関数であり、回帰問題で一般的に用いられます。  

In [ ]:
layer = 3

W3 = sym_matrix("W", 2, 2, layer=layer)
b3 = sym_vector("b", 2, layer=layer)
a3 = sym_vector("a", 2, layer=layer)

y = sym_vector("y", 2, layer=layer)

a3_eq = z2*W3 + b3
y_eq = identity_sym(a3)

In [ ]:
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = \mathbf{{z}}^{{({layer-1})}}\,\mathbf{{W}}^{{({layer})}} + \mathbf{{b}}^{{({layer})}}$$"))

In [ ]:
display(Markdown(rf"$$\mathbf{{z}}^{{({layer-1})}} = {latex_matrix(z2)}$$"))
display(Markdown(rf"$$\mathbf{{W}}^{{({layer})}} = {latex_matrix(W3)}$$"))
display(Markdown(rf"$$\mathbf{{b}}^{{({layer})}} = {latex_matrix(b3)}$$"))
display(Markdown(rf"$$\mathbf{{a}}^{{({layer})}} = {latex_matrix(a3_eq)}$$"))

In [ ]:
subs3 = {
    z2[0]: float(z2_num[0]), z2[1]: float(z2_num[1]),
    W3[0,0]: 0.1, W3[0,1]: 0.3,
    W3[1,0]: 0.2, W3[1,1]: 0.4,
    b3[0]: 0.1, b3[1]: 0.2,
}

W3_num = W3.subs(subs3)
b3_num = b3.subs(subs3)
a3_num = (z2*W3 + b3).subs(subs3)
z3_num = identity_sym(a3_num)

In [ ]:
display(Markdown(rf"$${latex_matrix(a3)} = {latex_matrix(z2)}\,{latex_matrix(W3)} + {latex_matrix(b3)}$$"))
display(Markdown(rf"$${latex_matrix(a3_num.evalf(3))} = {latex_matrix(z2_num.evalf(3))}\,{latex_matrix(W3_num)} + {latex_matrix(b3_num)}$$"))
display(Markdown(rf"$${latex_matrix(a3_num.evalf(3))} = {latex_matrix((z2_num * W3_num).evalf(3))} + {latex_matrix(b3_num)}$$"))

In [ ]:
subs3_disp = make_subs_disp(subs3, fmt="{:.2f}")
eqs_A3 = []
for j in range(a3_eq.cols):
    lhs = float(a3_num[0, j])
    rhs = sp.expand(a3_eq[0, j]).subs(subs3_disp)  # 右辺は未評価のまま
    eqs_A3.append(sp.Eq(sp.Symbol(f"{lhs:.3f}"), rhs, evaluate=False))

display(Markdown("$$" + r"\begin{aligned}"
                + r"\\ ".join(latex(e, mul_symbol="dot") for e in eqs_A3)
                + r"\end{aligned}" + "$$"))

In [ ]:
display(Markdown(rf"""
$$
\mathrm{{a}} = \mathrm{{z}} \, \mathrm{{W}} + \mathrm{{b}} = {latex_matrix(a3_num.evalf(3))},\qquad
\mathrm{{Y}} = \mathrm{{identity}}(\mathrm{{a}}) = \mathrm{{a}} = {latex_matrix(z3_num.evalf(3))}
$$
"""))

### 実装のまとめ

これらの処理は、重みとバイアスを辞書型の変数 network に初期化する init_network() 関数と、入力から出力への信号伝達を行う forward() 関数にまとめて実装されます。  

この入力から出力方向への一連の伝達処理を、順伝播 (forward propagation) と呼びます。NumPy の多次元配列を利用することで、ニューラルネットワークの順伝播処理を非常に効率的に実装することが可能となります。  

## 出力層の設計

この節では、ニューラルネットワークの最後の層である出力層 (第 3 層) 活性化関数の選び方について解説します。出力層で用いる活性化関数は、そのネットワークが解こうとしている問題の種類によって異なります。  

**問題の種類と活性化関数の選択**  

機械学習の問題は、大きく「分類問題」と「回帰問題」に分けられます。  

* 回帰問題 (Regression Problem)：連続的な数値の予測を行う問題です。  

    * 活性化関数：**恒等関数 (Identity Function)** を用いるのが一般的  

* 分類問題 (Classification Problem)：データがどのクラスに属するかを分類する問題です。  

    * 活性化関数：**ソフトマックス関数 (Softmax Function)** を用いるのが一般的  

**恒等関数 (Identity Function)**  

恒等関数は、入力された信号をそのまま出力する関数です。  

* 特徴：入力された情報を変換せずに次へ渡すため、出力層で恒等関数を用いる場合、信号の総和がそのままネットワークの最終出力となります。  

**ソフトマックス関数 (Softmax Function)**  

分類問題で主に用いられるソフトマックス関数は、次の数式で表されます。  

$$y_k = \frac{\exp(a_k)}{\sum_{i=1}^n \exp(a_i)} \quad \tag{3.10}$$

ここで $a_k$ は出力層の $k$ 番目のニューロンへの入力信号です。

**ソフトマックス関数の特徴**

1. 出力は 0 から 1 の実数になる：ソフトマックス関数の出力 $y_k$ は、0 から 1.0 の間の実数になります。  

2. 出力の総和は 1 になる：出力層のすべての要素の和は必ず 1 になります。  

3. 確率としての解釈：この「総和が 1 になる」という性質により、ソフトマックス関数の出力を **「確率」** として解釈することが可能です。これにより、「99.1 % の確率でクラス 2 である」といった、確率的な対応ができるようになります。  

**実装上の注意点**

ソフトマックス関数の実装においては、指数関数 $\exp(a_k)$ の計算時に、入力 $a_k$ の値が大きすぎると、計算結果が無限大 (`inf`) となり、オーバーフローが発生する危険性があります。  

In [ ]:
a = np.array([[1010], [1000], [990]])
np.exp(a) / np.sum(np.exp(a))

**オーバーフロー対策対策：**  

この問題を回避するため、入力信号 $a_k$ の各要素から、その中の最大値 $c$ を引くことで、指数関数の計算結果が非常に大きくなることを防ぎます。

$$y_k = \frac{\exp(a_k)}{\sum_{i=1}^n \exp(a_i)} = \frac{\exp(a_k + C')}{\sum_{i=1}^n \exp(a_i + C')} \quad \tag{3.11}$$

In [ ]:
c = np.max(a)
print(a - c)
np.exp(a - c) / np.sum(np.exp(a - c))

**出力層のニューロンの数**

出力層のニューロンの数は、解くべき分類問題のクラスの数に設定するのが一般的です。  

* 例：数字の 0 から 9 の 10 クラス分類を行う問題 (MNIST など) では、出力層のニューロンは 10 個に設定されます。  

**推論時のソフトマックス関数の省略**  

ニューラルネットワークの推論を行う際、分類クラスを決定する目的であれば、出力層のソフトマックス関数を省略するのが一般的です。  

* 理由：ソフトマックス関数を適用しても、各要素の大小関係は変わりません (指数関数が単調増加するため)。そのため、最も大きいスコア (Softmax を適用する前の値) を持つニューロンを選ぶだけで、分類結果は Softmax を適用した場合と同じになります。指数関数の計算は計算コストがかかるため、高速化のために省略されます。

## 手書き数字認識

**ニューラルネットワークの応用と高速化**  

この節では、学習済みのパラメータを用いて手書き数字の画像データセットである MNIST に対するニューラルネットワークの「推論処理」(順方向伝播) を実装します。また、大規模なデータを効率的に処理するためのバッチ処理について紹介します。


1. MNIST データセットの概要  

* データの内容：MNIST は、0 から 9 までの手書き数字の画像セットです。  
  * 訓練画像が 60,000 枚、テスト画像が 10,000 枚用意されています。  
  * 画像データは 28×28 ピクセルのグレー画像 (1チャンネル) で、各ピクセルは 0 から 255 までの値を取ります。  

* 一般的な使い方：訓練画像を使って学習を行い、学習したモデルの性能をテスト画像で評価します。  

* データの読み込み：`load_mnist()` 関数 (`dataset/mnist.py`) を使用することで、MNIST データを簡単に読み込むことができます。  
  * `normalize=True` を設定すると、ピクセル値が 0.0～1.0 の範囲に正規化されます。このような決まった変換を前処理と呼びます。  
  * `flatten=True` を設定すると、入力画像 (28×28) は 784 個の要素からなる 1 次元配列 (1 列) として格納されます。  

In [ ]:
def img_show(img):
    pil_img = Image.fromarray(np.uint8(img))
    pil_img.show()

(x_train, t_train), (x_test, t_test) = load_mnist(flatten=True, normalize=False)

img = x_train[0]
label = t_train[0]
print(label)  # 5

print(img.shape)  # (784,)
img = img.reshape(28, 28)  # 形状を元の画像サイズに変形
print(img.shape)  # (28, 28)

img_show(img)

2. ニューラルネットワークによる推論処理  
学習済みネットワークを用いて、テスト画像データに対して推論処理 (分類) を行います。  

* ネットワークの構成：この推論処理に使用するネットワークは、入力層 784 個、出力層 10 個 (0 から 9 の 10 クラス分類)、そして間に 2 つの隠れ層 (50 個および 100 個のニューロン) を持つ構成を想定しています。  

* 推論の手順：  
  1. 画像データを1枚ずつ取り出し、学習済みの重みパラメータを使用して順伝播処理 (`predict()`) を実行します。  
  2. `predict()` 関数の結果は、各ラベルの確率 (ソフトマックス関数の出力) を NumPy 配列として出力します。  
  3. 配列の中で最も大きな値を持つ要素のインデックス (`np.argmax()` で取得) を予測結果とします。  

* ソフトマックス関数の役割：推論時において、分類クラスを決定する目的であれば、ソフトマックス関数は計算コスト削減のために省略されるのが一般的です。なぜなら、ソフトマックス関数を適用しても、最大値を持つニューロンの場所 (大小関係) は変わらないからです。

3. バッチ処理の導入  

画像を 1 枚ずつ処理するのではなく、複数枚まとめて処理するバッチ処理を導入することで、計算効率を大幅に向上させます。  

* バッチ (batch)：まとまりのある入力データの束のことを指します。  

* データ形状の変化：  
  * 画像を 1 枚だけ入力する場合、入力データの形状は (784,)、出力は (10,) です。
  * 例えば 100 枚の画像をまとめて入力する場合、入力データの形状は (100, 784)、出力は (100, 10) となり、100 枚分の推論結果が一度に出力されます。  

* バッチ処理の利点：
  * 数値計算ライブラリの多くは、大きな配列の計算を効率良く処理できるような高度な最適化が行われています。そのため、バッチ処理によって、1 枚あたりの処理時間を大幅に短縮できます。
  * データの読み込みに対して演算の割合を多くできるため、データ転送がボトルネックとなる場合の負荷を軽減できます。

In [ ]:
def get_data():
    (x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, flatten=True, one_hot_label=False)
    return x_test, t_test


def init_network():
    with open("../ch03/sample_weight.pkl", 'rb') as f:
        network = pickle.load(f)
    return network


def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']

    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y


x, t = get_data()
network = init_network()
accuracy_cnt = 0
for i in range(len(x)):
    y = predict(network, x[i])
    p= np.argmax(y) # 最も確率の高い要素のインデックスを取得
    if p == t[i]:
        accuracy_cnt += 1

print("Accuracy:" + str(float(accuracy_cnt) / len(x)))

## まとめ

ニューラルネットワーク推論について  

この章（第 3 章）では、ニューラルネットワークの基礎的な構造と、入力から出力への信号伝達、すなわち**順伝播 (Forward Propagation)**について学習しました。  

1. 活性化関数の役割 (パーセプトロンとの違い)  
ニューラルネットワークを構成する各ニューロンは、パーセプトロンと同じく階層的に信号を伝達します。しかし、両者には活性化関数に大きな違いがあります。  

* ニューラルネットワーク：活性化関数としてシグモイド関数や ReLU 関数のような「滑らかに変化する関数」を利用します。これにより、ニューロン間を連続的な実数値の信号が流れるようになります。

* パーセプトロン：信号が急に変化するステップ関数 (階段関数) を利用していました。  
この「滑らかさ」の違いが、後の章で学ぶニューラルネットワークの学習 (微分を使ったパラメータ更新) において非常に重要となります。  

2. NumPy による効率的な実装  

* 多次元配列の活用：NumPy の多次元配列を効果的に利用することで、ニューラルネットワークの計算を行列の積としてまとめて処理できます。これにより、ニューラルネットワークの実装を効率良く行うことができました。  

3. 出力層の設計と問題の種類  
出力層で使用する活性化関数は、そのネットワークが解くべき問題の性質によって使い分けられます。

* 回帰問題：恒等関数を一般的に利用します。  

* 分類問題：ソフトマックス関数を一般的に利用します。また、分類問題の場合、出力層のニューロンの数は分類するクラスの数に設定するのが一般的です。  

4. バッチ処理による高速化  

* バッチ処理：入力データのまとまりをバッチと呼びます。

* 利点：データをバッチ単位で推論処理を行うことで、数値計算ライブラリの高度な最適化を利用でき、計算を高速に行うことができます。
この第 3 章で学んだ順伝播の知識は、次章以降で学ぶ学習 (最適な重みパラメータを自動で獲得するプロセス) の土台となります。